# Preprocess Telco Dataset with Column Type Detection

This notebook loads the cleaned Telco customer churn dataset, detects column types, and applies appropriate preprocessing:
- **Categorical columns**: One-hot encoding
- **Numeric columns**: Standard scaling
- All features are combined and scaled for clustering analysis.

In [1]:
import pandas as pd
import numpy as np
from typing import Dict, Tuple, List, Any, Optional
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import warnings

# Define file paths
input_file = r"C:\Users\weezh\OneDrive\Desktop\pyhercules\dataset\3rd exploration mdl implimentation\Telco_customer_churn_cleaned.csv"
output_file = r"C:\Users\weezh\OneDrive\Desktop\pyhercules\dataset\3rd exploration mdl implimentation\Telco_customer_churn_preprocessed.csv"

print("Loading cleaned dataset...")
df = pd.read_csv(input_file)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Loading cleaned dataset...
Dataset shape: (7043, 24)
Columns: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [2]:
def detect_column_types(df: pd.DataFrame) -> Dict[str, str]:
    """
    Automatically detects column types as 'numeric', 'categorical', or 'text'.
    
    Returns:
        Dict mapping column names to detected types ('numeric', 'categorical', 'text')
    """
    types = {}
    
    for col in df.columns:
        # 1. Try to detect if column is actually numeric (even if stored as object)
        if pd.api.types.is_numeric_dtype(df[col]):
            types[col] = 'numeric'
            continue
        
        # Try converting to numeric to see if it's numeric data stored as strings
        try:
            # Attempt conversion - if most values convert successfully, treat as numeric
            numeric_converted = pd.to_numeric(df[col], errors='coerce')
            non_null_ratio = numeric_converted.notna().sum() / len(df[col])
            
            # If >80% of non-null values can be converted to numeric, treat as numeric
            if non_null_ratio > 0.8:
                types[col] = 'numeric'
                continue
        except:
            pass
        
        # 2. For object/string columns, determine if categorical or text
        if df[col].dtype == 'object' or df[col].dtype.name == 'category':
            n_unique = df[col].nunique()
            n_total = len(df[col].dropna())
            
            if n_total == 0:
                types[col] = 'text'  # Safe default for empty columns
                continue
                
            cardinality_ratio = n_unique / n_total
            avg_length = df[col].astype(str).str.len().mean()
            
            # Categorical criteria: low cardinality OR low unique count, AND short strings
            is_low_cardinality = cardinality_ratio < 0.05 or n_unique < 50
            is_short = avg_length < 30
            
            if is_low_cardinality and is_short:
                types[col] = 'categorical'
            else:
                types[col] = 'text'
        else:
            # Fallback for other dtypes (datetime, etc.)
            types[col] = 'text'
    
    return types

# Detect column types
print("\nDetecting column types...")
column_types = detect_column_types(df)

# Display detected types
print("\nDetected column types:")
for col_type in ['numeric', 'categorical', 'text']:
    cols = [col for col, ctype in column_types.items() if ctype == col_type]
    if cols:
        print(f"\n{col_type.upper()} ({len(cols)} columns):")
        for col in cols:
            print(f"  - {col}")


Detecting column types...

Detected column types:

NUMERIC (6 columns):
  - Tenure Months
  - Monthly Charges
  - Total Charges
  - Churn Value
  - Churn Score
  - CLTV

CATEGORICAL (18 columns):
  - Gender
  - Senior Citizen
  - Partner
  - Dependents
  - Phone Service
  - Multiple Lines
  - Internet Service
  - Online Security
  - Online Backup
  - Device Protection
  - Tech Support
  - Streaming TV
  - Streaming Movies
  - Contract
  - Paperless Billing
  - Payment Method
  - Churn Label
  - Churn Reason


In [3]:
def preprocess_mixed_data(df: pd.DataFrame, 
                          column_types: Dict[str, str],
                          variable_metadata: Optional[Dict[str, Dict[str, Any]]] = None
                          ) -> Tuple[np.ndarray, List[str], Dict[str, Dict[str, Any]]]:
    """
    Preprocesses mixed data types (numeric, categorical, text) into a unified numerical representation.
    
    Args:
        df: Input DataFrame
        column_types: Dict mapping column names to types ('numeric', 'categorical', 'text')
        variable_metadata: Optional existing metadata to preserve
        
    Returns:
        Tuple of (processed_data_array, column_names_list, metadata_dict)
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    
    processed_parts = []
    final_column_names = []
    metadata_out = variable_metadata.copy() if variable_metadata else {}
    
    for col, col_type in column_types.items():
        if col not in df.columns:
            continue
            
        if col_type == 'numeric':
            # Keep numeric columns as-is, but handle missing/invalid values
            # Convert to numeric, coercing errors to NaN, then fill NaN with column mean
            col_data = pd.to_numeric(df[col], errors='coerce')
            col_mean = col_data.mean()
            col_data = col_data.fillna(col_mean if not pd.isna(col_mean) else 0)
            processed_parts.append(col_data.values.reshape(-1, 1))
            final_column_names.append(col)
            if col not in metadata_out:
                metadata_out[col] = {'name': col, 'type': 'numeric', 'source_column': col}
                
        elif col_type == 'categorical':
            # One-hot encode categorical columns
            # Fill NaN with a placeholder category
            col_data = df[col].fillna('_MISSING_')
            encoded = pd.get_dummies(col_data, prefix=col, drop_first=False, dtype=float)
            if encoded.shape[1] > 0:  # Only add if there are columns after encoding
                processed_parts.append(encoded.values)
                for new_col in encoded.columns:
                    final_column_names.append(new_col)
                    category_value = new_col.replace(f"{col}_", "")
                    metadata_out[new_col] = {
                        'name': new_col,
                        'type': 'categorical_encoded',
                        'source_column': col,
                        'category': category_value
                    }
        
        elif col_type == 'text':
            # TF-IDF + SVD for text columns
            texts = df[col].fillna('').astype(str)
            
            # Skip if all texts are empty
            if texts.str.strip().eq('').all():
                continue
            
            try:
                # TF-IDF vectorization
                vectorizer = TfidfVectorizer(
                    max_features=100,
                    stop_words='english',
                    min_df=1,
                    max_df=0.95
                )
                tfidf_matrix = vectorizer.fit_transform(texts)
                
                # SVD for dimensionality reduction
                n_components = min(10, tfidf_matrix.shape[1] - 1, tfidf_matrix.shape[0] - 1)
                if n_components > 0:
                    svd = TruncatedSVD(n_components=n_components, random_state=42)
                    text_features = svd.fit_transform(tfidf_matrix)
                    
                    processed_parts.append(text_features)
                    for i in range(text_features.shape[1]):
                        new_col_name = f"{col}_text_dim{i}"
                        final_column_names.append(new_col_name)
                        metadata_out[new_col_name] = {
                            'name': new_col_name,
                            'type': 'text_embedding',
                            'source_column': col,
                            'method': 'tfidf_svd',
                            'dimension': i
                        }
            except Exception as e:
                warnings.warn(f"Error processing text column '{col}': {e}. Skipping.")
                continue
    
    if not processed_parts:
        raise ValueError("No valid columns to process after preprocessing.")
    
    # Combine all processed parts horizontally
    combined_data = np.hstack(processed_parts)
    
    return combined_data, final_column_names, metadata_out

# Apply preprocessing
print("\nApplying preprocessing (one-hot encoding for categorical, keeping numeric)...")
processed_data, processed_column_names, metadata = preprocess_mixed_data(df, column_types)

print(f"\nPreprocessed data shape: {processed_data.shape}")
print(f"Number of features after encoding: {len(processed_column_names)}")
print(f"\nFirst 10 feature names: {processed_column_names[:10]}")


Applying preprocessing (one-hot encoding for categorical, keeping numeric)...

Preprocessed data shape: (7043, 72)
Number of features after encoding: 72

First 10 feature names: ['Gender_Female', 'Gender_Male', 'Senior Citizen_No', 'Senior Citizen_Yes', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'Tenure Months', 'Phone Service_No']


In [4]:
# Apply Standard Scaling to all features
print("\nApplying StandardScaler to all features...")
scaler = StandardScaler()
scaled_data = scaler.fit_transform(processed_data)

print(f"Scaled data shape: {scaled_data.shape}")
print(f"Mean of scaled data (should be ~0): {scaled_data.mean():.6f}")
print(f"Std of scaled data (should be ~1): {scaled_data.std():.6f}")


Applying StandardScaler to all features...
Scaled data shape: (7043, 72)
Mean of scaled data (should be ~0): -0.000000
Std of scaled data (should be ~1): 1.000000


In [5]:
# Save the preprocessed dataset
print(f"\nSaving preprocessed dataset to: {output_file}")

# Create DataFrame with scaled data and feature names
df_preprocessed = pd.DataFrame(scaled_data, columns=processed_column_names)

# Save to CSV
df_preprocessed.to_csv(output_file, index=False)

print(f"✓ Preprocessed dataset saved successfully!")
print(f"  - Original columns: {len(df.columns)}")
print(f"  - Preprocessed features: {len(processed_column_names)}")
print(f"  - Total rows: {df_preprocessed.shape[0]}")
print(f"  - All features are standardized (mean=0, std=1)")


Saving preprocessed dataset to: C:\Users\weezh\OneDrive\Desktop\pyhercules\dataset\3rd exploration mdl implimentation\Telco_customer_churn_preprocessed.csv
✓ Preprocessed dataset saved successfully!
  - Original columns: 24
  - Preprocessed features: 72
  - Total rows: 7043
  - All features are standardized (mean=0, std=1)


In [6]:
# Preview the preprocessed dataset
print("\nFirst 5 rows of preprocessed dataset:")
df_preprocessed.head()


First 5 rows of preprocessed dataset:


,Gender_Female,Gender_Male,Senior Citizen_No,Senior Citizen_Yes,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,Tenure Months,Phone Service_No,...,Churn Reason_Limited range of services,Churn Reason_Long distance charges,Churn Reason_Moved,Churn Reason_Network reliability,Churn Reason_Poor expertise of online support,Churn Reason_Poor expertise of phone support,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction,Churn Reason__MISSING_
0,-0.990532,0.990532,0.439916,-0.439916,0.966622,-0.966622,0.548093,-0.548093,-1.236724,-0.327438,...,-0.079288,-0.079288,-0.087076,-0.121826,-0.05201,-0.053365,-0.118789,-0.121224,-0.11313,-1.663829
1,1.009559,-1.009559,0.439916,-0.439916,0.966622,-0.966622,-1.824507,1.824507,-1.236724,-0.327438,...,-0.079288,-0.079288,11.484198,-0.121826,-0.05201,-0.053365,-0.118789,-0.121224,-0.11313,-1.663829
2,1.009559,-1.009559,0.439916,-0.439916,0.966622,-0.966622,-1.824507,1.824507,-0.992402,-0.327438,...,-0.079288,-0.079288,11.484198,-0.121826,-0.05201,-0.053365,-0.118789,-0.121224,-0.11313,-1.663829
3,1.009559,-1.009559,0.439916,-0.439916,-1.034530,1.034530,-1.824507,1.824507,-0.177995,-0.327438,...,-0.079288,-0.079288,11.484198,-0.121826,-0.05201,-0.053365,-0.118789,-0.121224,-0.11313,-1.663829
4,-0.990532,0.990532,0.439916,-0.439916,0.966622,-0.966622,-1.824507,1.824507,0.677133,-0.327438,...,-0.079288,-0.079288,-0.087076,-0.121826,-0.05201,-0.053365,-0.118789,-0.121224,-0.11313,-1.663829


In [7]:
# Summary statistics
print("\nSummary of preprocessing:")
print(f"  - Numeric columns: {sum(1 for t in column_types.values() if t == 'numeric')}")
print(f"  - Categorical columns: {sum(1 for t in column_types.values() if t == 'categorical')}")
print(f"  - Text columns: {sum(1 for t in column_types.values() if t == 'text')}")
print(f"\nFeature expansion:")
print(f"  - Before one-hot encoding: {len(column_types)} columns")
print(f"  - After one-hot encoding: {len(processed_column_names)} features")
print(f"  - Expansion ratio: {len(processed_column_names)/len(column_types):.2f}x")


Summary of preprocessing:
  - Numeric columns: 6
  - Categorical columns: 18
  - Text columns: 0

Feature expansion:
  - Before one-hot encoding: 24 columns
  - After one-hot encoding: 72 features
  - Expansion ratio: 3.00x
